# Job offers: internship and interview preparation

**Binary Logistic Regression** · a worked example from the StatNex teaching guide

**The story.** A careers office followed 26 graduates through their job search. It recorded whether each had done an internship and how many hours they spent preparing for interviews, and whether they got a job offer.

**The question.** How much do an internship and interview preparation raise the chance of getting an offer?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/binary_logistic_regression.csv` (26 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Internship,Interview prep (hours),Job offer
1.0,0.0,2.0,0.0
2.0,1.0,17.0,1.0
3.0,0.0,10.0,0.0
4.0,0.0,2.0,0.0
5.0,0.0,13.0,1.0
6.0,0.0,22.0,1.0
7.0,0.0,7.0,0.0
8.0,0.0,18.0,1.0
9.0,0.0,17.0,1.0
10.0,0.0,3.0,0.0
11.0,1.0,6.0,0.0
12.0,1.0,20.0,1.0
13.0,1.0,15.0,1.0
14.0,1.0,4.0,0.0
15.0,0.0,5.0,0.0
16.0,0.0,26.0,1.0
17.0,0.0,21.0,0.0
18.0,1.0,2.0,1.0
19.0,0.0,19.0,1.0
20.0,1.0,9.0,1.0
21.0,1.0,8.0,1.0
22.0,1.0,22.0,1.0
23.0,1.0,16.0,1.0
24.0,1.0,17.0,1.0
25.0,0.0,4.0,0.0
26.0,0.0,15.0,1.0

These cells save `collinear_columns.py`, `firth.py`, `hosmer_lemeshow.py`, `logistic_roc.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile collinear_columns.py
"""Which explanatory variables are exact linear combinations of the ones before them.

Such a variable adds no information and makes the model impossible to estimate. Each
column is centred and scaled to length 1, then regressed on the columns already kept;
if nothing is left over (squared residual length at most 1e-10) the column is
redundant. A constant column is redundant too. The *later* member of a redundant pair
is the one reported, so a variable listed first survives.
"""

import numpy as np


def collinear_columns(values, names, tolerance=1e-10):
    centred = values - values.mean(axis=0)
    scale = np.linalg.norm(centred, axis=0)
    scale[scale == 0] = 1.0
    centred = centred / scale
    dropped, kept = [], []
    for j, name in enumerate(names):
        column = centred[:, j]
        if kept:
            basis = centred[:, kept]
            column = column - basis @ np.linalg.lstsq(basis, column, rcond=None)[0]
        if column @ column <= tolerance or values[:, j].max() == values[:, j].min():
            dropped.append(name)
        else:
            kept.append(j)
    return dropped

In [ ]:
%%writefile firth.py
"""Firth's penalised logistic regression, for a fit that runs into separation.

Under separation the ordinary maximum-likelihood coefficient has no finite value.
Firth (1993) adds half the log-determinant of the information to the log-likelihood
(Jeffreys' prior), which keeps every estimate finite. R's logistf does this; Python's
statistics libraries do not, so it is written out.
"""

import numpy as np
import pandas as pd
from scipy import special, stats


def penalised(y, design, beta):
    """(penalised log-likelihood, fitted p, weights p(1-p), information X'WX)."""
    p = special.expit(design @ beta)
    w = p * (1 - p)
    information = design.T @ (design * w[:, None])
    sign, logdet = np.linalg.slogdet(information)
    with np.errstate(divide="ignore"):
        loglik = np.sum(np.where(y == 1, np.log(p), np.log1p(-p)))
    return (loglik + 0.5 * logdet if sign > 0 else -np.inf), p, w, information


def firth_fit(y, design, fixed=None, max_iter=200):
    """(coefficients, covariance, penalised log-likelihood); ``fixed`` holds one at 0.

    Newton steps on the modified score X'(y - p + h(1/2 - p)), h the hat diagonal,
    capped at 5 and halved until the penalised likelihood rises, as logistf does.
    """
    columns = design.shape[1]
    beta = np.zeros(columns)
    free = np.ones(columns, dtype=bool)
    if fixed is not None:
        free[fixed] = False
    value, p, w, information = penalised(y, design, beta)
    for _ in range(max_iter):
        covariance = np.linalg.inv(information)
        hat = w * np.einsum("ij,jk,ik->i", design, covariance, design)
        score = design.T @ (y - p + hat * (0.5 - p))
        step = np.zeros(columns)
        step[free] = np.linalg.solve(information[np.ix_(free, free)], score[free])
        largest = np.abs(step).max()
        if largest > 5:
            step *= 5 / largest
        for _ in range(30):
            candidate = penalised(y, design, beta + step)
            if candidate[0] >= value - 1e-12:
                break
            step /= 2
        beta = beta + step
        value, p, w, information = candidate
        if np.abs(step).max() < 1e-10:
            break
    return beta, np.linalg.inv(information), value


def firth_table(y, design, names):
    """b, SE, and the penalised likelihood-ratio test of each coefficient against 0.

    The test refits with that coefficient held at 0, the penalty still over the full
    model -- logistf's test, which does not lean on the Wald approximation that
    separation breaks.
    """
    beta, covariance, full = firth_fit(y, design)
    rows = []
    for j in range(design.shape[1]):
        restricted = firth_fit(y, design, fixed=j)[2]
        chi2 = max(0.0, 2 * (full - restricted))
        rows.append([beta[j], np.sqrt(covariance[j, j]), chi2, stats.chi2.sf(chi2, 1)])
    return pd.DataFrame(rows, index=names, columns=["b", "SE", "chi-square", "P"])

In [ ]:
%%writefile hosmer_lemeshow.py
"""Hosmer and Lemeshow's goodness-of-fit test, with the app's grouping.

The cases are put in order of predicted probability and cut into ten groups, and each
group's observed events are set against the sum of its predicted probabilities
(Hosmer, Lemeshow and Sturdivant 2013, section 5.2.2).

The textbook leaves ties open, so the rule is this: group k ends at the predicted
probability of the case at position ceil(k N / 10) in that order, and holds every case
above the previous group's end and at or below its own. Cases with equal predicted
probabilities are therefore never split, and ties can leave fewer than ten groups; the
degrees of freedom are the groups actually formed less 2. A row of grouped data counts
as `trials` cases and is never split either. It is Stata's `estat gof, group(10)`.
"""

import numpy as np
import pandas as pd
from scipy import stats


def hosmer_lemeshow(events, trials, fitted, groups=10):
    """The groups as a table, and the chi-square, its df and its P."""
    events, trials, fitted = (np.asarray(a, dtype=float) for a in (events, trials, fitted))
    order = np.argsort(fitted, kind="stable")
    count = np.cumsum(trials[order])
    total = count[-1]
    # The first case whose running count reaches k N / groups -- compared as
    # k N <= groups x count, so that no fraction is rounded on the way.
    ends = np.unique([fitted[order][np.argmax(count * groups >= k * total)]
                      for k in range(1, groups + 1)])
    group = np.searchsorted(ends, fitted) + 1
    table = pd.DataFrame({"group": group, "n": trials, "observed": events,
                          "expected": trials * fitted}).groupby("group").sum()
    table["upper end"] = ends
    chi2 = ((table["observed"] - table["expected"]) ** 2
            / (table["expected"] * (1 - table["expected"] / table["n"]))).sum()
    df = len(table) - 2
    return table, chi2, df, stats.chi2.sf(chi2, df)

In [ ]:
%%writefile logistic_roc.py
"""Area under the ROC curve with DeLong's standard error, and the curve itself.

scikit-learn would give the area but not its standard error, and SciPy has neither,
so both are written out. The area is the Mann-Whitney statistic: the share of
(positive, negative) pairs in which the positive case scores higher, a tie counting
half. DeLong's standard error comes from each case's own share of those comparisons
(its "structural component"), computed here from midranks.
"""

import numpy as np
import pandas as pd
from scipy import stats


def delong(positive, negative):
    """The area under the curve and its standard error."""
    m, n = len(positive), len(negative)
    rank_all = stats.rankdata(np.concatenate([positive, negative]))
    area = (rank_all[:m].sum() - m * (m + 1) / 2) / (m * n)
    v10 = (rank_all[:m] - stats.rankdata(positive)) / n
    v01 = 1 - (rank_all[m:] - stats.rankdata(negative)) / m
    return area, np.sqrt(v10.var(ddof=1) / m + v01.var(ddof=1) / n)


def roc_points(positive, negative):
    """One row per cut-off: a case is called positive when its score is >= the cut-off.

    The cut-offs are every distinct score, plus one beyond the largest so the curve
    reaches (0, 0).
    """
    scores = np.unique(np.concatenate([positive, negative]))
    cutoffs = np.append(scores, scores[-1] + 1)
    return pd.DataFrame({
        "cutoff": cutoffs,
        "FPF": [(negative >= c).mean() for c in cutoffs],
        "TPF": [(positive >= c).mean() for c in cutoffs],
    })

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from collinear_columns import collinear_columns
from firth import firth_table
from logistic_roc import delong, roc_points
from hosmer_lemeshow import hosmer_lemeshow

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# The response is coded 0/1. Text in a numeric column counts as missing.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Job offer"
predictors = ["Internship", "Interview prep (hours)"]
frame = data[[response, *predictors]].apply(pd.to_numeric, errors="coerce")
max_iter = 50
confidence = 95 / 100

## Cases

In [ ]:
y_ok = frame[response].notna()
x_ok = frame[predictors].notna().all(axis=1)
summary = pd.DataFrame(
    {"n": [(y_ok & x_ok).sum(), (~y_ok & x_ok).sum(), (y_ok & ~x_ok).sum(),
           (~y_ok & ~x_ok).sum(), len(frame)]},
    index=["valid", "response missing", "explanatory missing", "both missing", "total"],
)
summary["%"] = summary["n"] / len(frame)
print(summary)
valid = frame[y_ok & x_ok]
y = valid[response].to_numpy(float)
n = len(y)
response_summary = pd.DataFrame({"n": [(y == 0).sum(), (y == 1).sum(), n]},
                                index=[f"{response}=0", f"{response}=1", "total"])
response_summary["%"] = response_summary["n"] / n
print()
print(response_summary)

## Descriptive statistics and correlations, overall and by outcome

In [ ]:
groups = {"all": valid, f"{response}=0": valid[y == 0], f"{response}=1": valid[y == 1]}
descriptive = pd.concat({
    label: pd.DataFrame({"n": len(part), "mean": part[predictors].mean(),
                         "variance": part[predictors].var(ddof=1), "SD": part[predictors].std(ddof=1),
                         "min": part[predictors].min(), "max": part[predictors].max()})
    for label, part in groups.items()
})
correlation = pd.concat({label: part[predictors].corr() for label, part in groups.items()})
print()
print(descriptive)
print()
print(correlation)

## Variables that are linear combinations of others

In [ ]:
values = valid[predictors].to_numpy(float)
dropped = collinear_columns(values, predictors)
print("\nLinearly dependent:", dropped or "none")
# They are left out of the model.
predictors = [name for name in predictors if name not in dropped]
values = valid[predictors].to_numpy(float)

## Variable selection

In [ ]:
intercept = True
# Every explanatory variable goes into the model, in one step.
chosen = list(range(len(predictors)))
steps = [chosen]
previous_deviance = None
names = [predictors[j] for j in chosen]
labels = names + (["Constant"] if intercept else [])
# The constant's row has no standardised coefficient or "value" in the tables below.
no_value = [np.nan] if intercept else []
print("Selected:", names)

## The model

In [ ]:
# Maximum likelihood by iteratively reweighted least squares, run to a 1e-12 change in
# the deviance. The constant is the last column, as the app prints it.
design = values[:, chosen]
if intercept:
    design = np.column_stack([design, np.ones(n)])
binomial = sm.families.Binomial()
model = sm.GLM(y, design, family=binomial).fit(maxiter=max_iter, tol=1e-12)
deviance = model.deviance
if intercept:
    null_deviance = sm.GLM(y, np.ones(n), family=binomial).fit(maxiter=max_iter, tol=1e-12).deviance
else:
    # Without a constant the baseline is the model with no term at all: every p = 0.5.
    null_deviance = -2 * n * np.log(0.5)
probabilities = model.fittedvalues
predicted = (probabilities >= 0.5).astype(int)

# Pseudo R²: McFadden (labelled "R squared"), Cox-Snell and Nagelkerke.
cox_snell = 1 - np.exp(-(null_deviance - deviance) / n)
fit_summary = pd.DataFrame([{
    "-2LL": deviance, "AIC": deviance + 2 * design.shape[1],
    "McFadden": 1 - deviance / null_deviance, "Cox-Snell": cox_snell,
    "Nagelkerke": cox_snell / (1 - np.exp(-null_deviance / n)),
    "r(y, p)": np.corrcoef(y, probabilities)[0, 1], "misclassified": (predicted != y).mean(),
}])
print()
print(fit_summary.to_string(index=False))

# Likelihood-ratio tests. The model's is against the baseline. The last step's is SPSS's
# Omnibus "Step" row: the deviance of the model one step before the last minus the final
# model's, on as many df as variables moved. A variable removed makes it negative, and
# its P is that variable's test, P(chi-square > |chi-square|). At step 1 it is the model's.
chi2 = null_deviance - deviance
df = design.shape[1] - (1 if intercept else 0)
if previous_deviance is None:
    step_chi2, step_df = chi2, df
else:
    step_chi2, step_df = previous_deviance - deviance, abs(len(chosen) - len(steps[-2]))
model_tests = pd.DataFrame([[step_chi2, step_df, stats.chi2.sf(abs(step_chi2), step_df)],
                            [chi2, df, stats.chi2.sf(chi2, df)]],
                           index=[f"step {len(steps)}", "model"],
                           columns=["chi-square", "df", "P"])
print()
print(model_tests)

## Coefficients

In [ ]:
# Wald test: (b/SE)² on 1 df. The standardised coefficient is b times the SD of x
# dividing by n (not n - 1), as the app computes it.
z = stats.norm.ppf(0.5 + confidence / 2)
b, se = model.params, model.bse
standardised = np.append(b[: len(chosen)] * values[:, chosen].std(axis=0), no_value)
coefficients = pd.DataFrame({
    "b": b, "SE": se, "standardised": standardised, "lower": b - z * se, "upper": b + z * se,
    "odds ratio": np.exp(b), "OR lower": np.exp(b - z * se), "OR upper": np.exp(b + z * se),
    "Wald": (b / se) ** 2, "df": 1, "P": stats.chi2.sf((b / se) ** 2, 1),
}, index=labels)
covariance = pd.DataFrame(model.cov_params()[: len(chosen), : len(chosen)], index=names, columns=names)
print()
print(coefficients)

## Firth's method (only under separation)

In [ ]:
# Quasi-complete separation shows as a fitted probability pinned at 0 or 1, or a
# coefficient that is large and swamped by its own SE (|b| > 10, SE > 100|b|); the app
# warns then, and adds Firth's penalised estimates, which stay finite (firth.py).
fitted_p = model.fittedvalues
eps = 10 * np.finfo(float).eps  # R's glm.fit threshold for "numerically 0 or 1"
pinned = ((fitted_p < eps) | (fitted_p > 1 - eps)).any()
runaway = ((np.abs(b) > 10) & (se > 100 * np.abs(b))).any()
if (pinned or runaway) and np.isin(y, (0, 1)).all():
    firth = firth_table(y, design, labels)
    print()
    print(firth)
print()
print(covariance)

## Classification table

In [ ]:
# A case is predicted 1 when its fitted probability is at least 0.5.
classification = pd.crosstab(pd.Series(y.astype(int), name="observed"),
                             pd.Series(predicted, name="predicted")).reindex(
    index=[0, 1], columns=[0, 1], fill_value=0)
classification["hit rate"] = np.diag(classification) / classification.sum(axis=1)
hit_rate = (predicted == y).mean()
print()
print(classification)
print(f"overall hit rate: {hit_rate:.4f}")

## What the model predicts when every variable is 1

In [ ]:
simulation = pd.DataFrame({"b": b, "value": [1.0] * len(chosen) + no_value,
                           "odds ratio": np.append(np.exp(b[: len(chosen)]), no_value)},
                          index=labels)
simulation.loc[response] = [np.nan, 1 / (1 + np.exp(-b.sum())), np.nan]
print()
print(simulation)

## ROC curve of the fitted probabilities

In [ ]:
# The closest point is nearest the top-left corner; the farthest is furthest above the
# diagonal (largest TPF - FPF, Youden's index).
positive, negative = probabilities[y == 1], probabilities[y == 0]
area, area_se = delong(positive, negative)
roc_chi2 = ((area - 0.5) / area_se) ** 2
roc_auc = pd.DataFrame([[area, area_se, max(area - z * area_se, 0.0), min(area + z * area_se, 1.0), roc_chi2, 1,
                         stats.chi2.sf(roc_chi2, 1)]],
                       columns=["area", "SE", "lower", "upper", "chi-square", "df", "P"])
curve = roc_points(positive, negative)
closest = curve.loc[(curve["FPF"] ** 2 + (1 - curve["TPF"]) ** 2).idxmin()].copy()
closest["odds ratio"] = (closest["TPF"] / (1 - closest["TPF"])) / (closest["FPF"] / (1 - closest["FPF"]))
farthest = curve.loc[(curve["TPF"] - curve["FPF"]).idxmax()]
print()
print(roc_auc.to_string(index=False))
print("closest to (0, 1):", closest.to_dict())
print("farthest from the diagonal:", farthest.to_dict())

## Fitted probabilities

In [ ]:
predictions = pd.DataFrame({"no": np.arange(1, n + 1), "observed": y, "predicted": probabilities})
print()
print(predictions.to_string(index=False))

## Hosmer-Lemeshow goodness of fit

In [ ]:
# Ten groups by predicted probability (deciles); cases with equal probabilities stay in
# one group, so ties can leave fewer (hosmer_lemeshow.py). df = groups - 2. Only a 0/1
# response has events to count.
if np.isin(y, (0, 1)).all():
    hl_groups, hl_chi2, hl_df, hl_p = hosmer_lemeshow(y, np.ones(n), probabilities)
    hosmer_lemeshow_test = [hl_chi2, hl_df, hl_p, len(hl_groups)]
    print()
    print(hl_groups)
    print(f"Hosmer-Lemeshow chi-square {hl_chi2:.4f}, df {hl_df}, P {hl_p:.4f}")

## What the result says

Binary logistic regression models the chance of a yes/no outcome, here a job offer, for 26 graduates. The model as a whole is significant: the likelihood ratio test gives P = 0.0001. A P value is the chance of seeing a link this strong if there were no real link.

The odds ratio says how the odds of an offer multiply when a predictor rises by one. For interview preparation it is 1.39 for each extra hour (P = 0.012). For an internship it is 19.9, but P = 0.057, and its confidence interval, the range of values the data fit, runs from 0.91 to 433. That range includes 1, which means no effect.

So preparation clearly goes with more offers. An internship may help a lot, but 26 graduates are too few to say how much, or even to rule out no effect.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.